# 温度でセンサーのずれを予測できるか

統計から機械学習への入口。100 gのおもりを量る架空の実験で、別の日のずれを予測する。[実習記録と教材](../learning/2026-11-21-sensor-temperature-regression.md)

## 1. 結果を見る前に予想する

| 1日目の温度 | 20 ℃ | 25 ℃ | 30 ℃ |
| --- | ---: | ---: | ---: |
| 表示のずれ | −2 g | ＋1 g | ＋1 g |

別の日の22 ℃と28 ℃を予測するなら、温度で予測を変える方がよさそうか、平均で十分そうか。理由とともに答える。**ここでHiroさんの回答を待つ。答えてから次へ進む。**

## 学習データで計算を追う

基準では1日目のずれ3件の平均を使う。直線では、温度とずれそれぞれの平均からの差を取り、差の積の和を温度の差の二乗和で割ると傾きになる。切片は「ずれの平均 − 傾き × 温度の平均」。この直線は学習点で残差の二乗和を最小にする。

下のセルはこの計算と、同じデータを受け取ったscikit-learnの `fit` が求める値を照合する。

In [ ]:
from math import isclose
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

data_dir = Path("data/sensor-temperature")
if not data_dir.is_dir():
    data_dir = Path("../data/sensor-temperature")

train = pd.read_csv(data_dir / "train.csv")
temperature = train["temperature_c"]
offset = train["offset_g"]
dx = temperature - temperature.mean()
dy = offset - offset.mean()
training_mean = offset.mean()
manual_slope = (dx * dy).sum() / (dx ** 2).sum()
manual_intercept = training_mean - manual_slope * temperature.mean()

train_inputs = train[["temperature_c"]]
baseline = DummyRegressor(strategy="mean").fit(train_inputs, offset)
regression = LinearRegression().fit(train_inputs, offset)
slope = regression.coef_[0]
intercept = regression.intercept_

assert isclose(baseline.constant_.item(), training_mean)
assert isclose(slope, manual_slope)
assert isclose(intercept, manual_intercept)
display(train.assign(dx=dx, dy=dy, dx_times_dy=dx * dy, dx_squared=dx ** 2))
print(f"Training mean = {training_mean:.2f} g")
print(f"Slope = {slope:.2f} g/°C, intercept = {intercept:.2f} g")

## 2. 22 ℃の予測を1行で書く

`prediction_22 = None` を、`slope`、`intercept`、`temperature_c` を使って予測を返す式に書き換える。セルを実行し、`predict` と一致したら次へ進む。

In [ ]:
temperature_c = 22
prediction_22 = None  # この1行を自分で書き換える

library_prediction_22 = regression.predict(pd.DataFrame({"temperature_c": [temperature_c]}))[0]
prediction_checked = prediction_22 is not None and isclose(prediction_22, library_prediction_22)

if prediction_checked:
    print(f"22 °C: your formula and predict agree ({prediction_22:.2f} g).")
else:
    print("予測式を入力して、一致を確認してください。")

## 3. 別の日で評価する

予想を残し、前のセルで一致を確認してから実行する。どちらのモデルにも同じ2件を渡す。絶対誤差は「予測 − 観測」の絶対値。表と図を見て、どの位置でどちらが近かったかを確かめる。

In [ ]:
if globals().get("prediction_checked", False):
    evaluation = pd.read_csv(data_dir / "evaluation.csv")
    evaluation_inputs = evaluation[["temperature_c"]]
    comparison = evaluation.rename(columns={"offset_g": "observed_g"}).copy()
    comparison["baseline_g"] = baseline.predict(evaluation_inputs)
    comparison["linear_g"] = regression.predict(evaluation_inputs)
    observed = comparison["observed_g"]
    for model_name in ("baseline", "linear"):
        predictions = comparison[f"{model_name}_g"]
        comparison[f"{model_name}_abs_error_g"] = (predictions - observed).abs()
    display(comparison.round(2))
    for model_name in ("baseline", "linear"):
        mae = mean_absolute_error(observed, comparison[f"{model_name}_g"])
        print(f"{model_name} MAE = {mae:.2f} g")

    grid = pd.DataFrame({"temperature_c": np.linspace(20, 30, 101)})
    fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
    ax.plot(grid["temperature_c"], regression.predict(grid), label="Linear regression")
    ax.plot(grid["temperature_c"], baseline.predict(grid), "--", label="Mean baseline")
    ax.scatter(temperature, offset, marker="o", color="#163a61", label="Day 1 (fit)", zorder=3)
    ax.scatter(evaluation["temperature_c"], observed, marker="D", color="#9e2828", label="Day 2 (evaluation)", zorder=3)
    ax.set(xlabel="Temperature (°C)", ylabel="Sensor offset (g)")
    ax.legend()
    plt.show()
else:
    print("先に「22 ℃の予測を1行で書く」を実行してください。")

## 4. 自分の言葉で解釈する

**ここでHiroさんの回答を待つ。** 最初の予想に対して、表か図のどの点が判断を変えたか。この2件から、温度を使う判断をどこまで支持できるか。

- 実行前の予想
- 実行後の解釈